In [ ]:
# cell 1
import ast

import numpy as np
import pandas as pd

from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier

from sklearn.metrics import (
    precision_score,
    recall_score,
    roc_auc_score,
    accuracy_score,
    f1_score,
    confusion_matrix,
    classification_report
)

from imblearn.over_sampling import RandomOverSampler

In [ ]:
# cell 2
# Settings
DATA_DIR = r"..\Data\model_data"
RESULTS_DIR = r"..\Results\Model_selection"
RESULTS_OUTPUT =  r"..\Results"

SEED = 10

OUTCOME_PREFIXES = [
    "mask",
    "protective",
    "social_avoidance"
]

In [ ]:
# cell 3
# Helper functions
def load_train_test_data(prefix):
    X_train = pd.read_csv(f"{DATA_DIR}\\{prefix}_X_train.csv")
    X_test = pd.read_csv(f"{DATA_DIR}\\{prefix}_X_test.csv")

    y_train = pd.read_csv(f"{DATA_DIR}\\{prefix}_y_train.csv").values.ravel()
    y_test = pd.read_csv(f"{DATA_DIR}\\{prefix}_y_test.csv").values.ravel()

    return X_train, X_test, y_train, y_test


def prepare_train_test_data(X_train, X_test):
    X_train = X_train.copy()
    X_test = X_test.copy()

    bool_cols_train = X_train.select_dtypes(include=["bool"]).columns
    bool_cols_test = X_test.select_dtypes(include=["bool"]).columns

    if len(bool_cols_train) > 0:
        X_train[bool_cols_train] = X_train[bool_cols_train].astype(int)

    if len(bool_cols_test) > 0:
        X_test[bool_cols_test] = X_test[bool_cols_test].astype(int)

    X_train, X_test = X_train.align(
        X_test,
        join="left",
        axis=1,
        fill_value=0
    )

    return X_train, X_test


def load_best_params(model_name):
    comparison_path = f"{RESULTS_DIR}\\{model_name}_cv_comparison.csv"
    comparison_df = pd.read_csv(comparison_path)

    best_params = {}

    for _, row in comparison_df.iterrows():
        outcome = row["outcome"]
        params = ast.literal_eval(row["best_params"])
        best_params[outcome] = params

    return best_params


def evaluate_on_test_set(model, X_test, y_test):
    y_pred = model.predict(X_test)
    y_prob = model.predict_proba(X_test)[:, 1]

    metrics = {
        "precision": precision_score(y_test, y_pred, zero_division=0),
        "recall": recall_score(y_test, y_pred, zero_division=0),
        "roc_auc": roc_auc_score(y_test, y_prob),
        "accuracy": accuracy_score(y_test, y_pred),
        "f1": f1_score(y_test, y_pred, zero_division=0)
    }

    cm = confusion_matrix(y_test, y_pred)

    report = classification_report(
        y_test,
        y_pred,
        zero_division=0,
        output_dict=True
    )

    return metrics, cm, report

In [ ]:
# cell 4
# Model fitting functions
def fit_random_forest(X_train, y_train, params):
    model = RandomForestClassifier(
        n_estimators=250,
        bootstrap=True,
        random_state=SEED,
        n_jobs=-1,
        **params
    )

    model.fit(X_train, y_train)

    return model


def fit_xgboost(X_train, y_train, params):
    model = XGBClassifier(
        n_estimators=250,
        objective="binary:logistic",
        eval_metric="auc",
        random_state=SEED,
        tree_method="hist",
        **params
    )

    model.fit(X_train, y_train)

    return model

In [ ]:
# cell 5
# Final test-set validation
rf_best_params = load_best_params("rf")
xgb_best_params = load_best_params("xgb")

all_test_results = []
all_confusion_matrices = []
all_classification_reports = []

for prefix in OUTCOME_PREFIXES:
    print("\n" + "=" * 60)
    print(f"Final test-set validation for: {prefix}")

    X_train, X_test, y_train, y_test = load_train_test_data(prefix)

    # Prepare feature columns and data types
    X_train, X_test = prepare_train_test_data(X_train, X_test)

    # Upsample the complete training set only
    upsampler = RandomOverSampler(random_state=SEED)
    X_train_up, y_train_up = upsampler.fit_resample(X_train, y_train)

    if not isinstance(X_train_up, pd.DataFrame):
        X_train_up = pd.DataFrame(
            X_train_up,
            columns=X_train.columns
        )

    models = {
        "Random Forest": fit_random_forest(
            X_train=X_train_up,
            y_train=y_train_up,
            params=rf_best_params[prefix]
        ),
        "XGBoost": fit_xgboost(
            X_train=X_train_up,
            y_train=y_train_up,
            params=xgb_best_params[prefix]
        )
    }

    for model_name, model in models.items():
        metrics, cm, report = evaluate_on_test_set(
            model=model,
            X_test=X_test,
            y_test=y_test
        )

        result_row = {
            "model": model_name,
            "outcome": prefix,
            "upsample": True,
            "precision": metrics["precision"],
            "recall": metrics["recall"],
            "roc_auc": metrics["roc_auc"],
            "accuracy": metrics["accuracy"],
            "f1": metrics["f1"]
        }

        all_test_results.append(result_row)

        cm_row = {
            "model": model_name,
            "outcome": prefix,
            "tn": cm[0, 0],
            "fp": cm[0, 1],
            "fn": cm[1, 0],
            "tp": cm[1, 1]
        }

        all_confusion_matrices.append(cm_row)

        for class_label, class_metrics in report.items():
            if class_label in ["0", "1"]:
                report_row = {
                    "model": model_name,
                    "outcome": prefix,
                    "class": class_label,
                    "precision": class_metrics["precision"],
                    "recall": class_metrics["recall"],
                    "f1": class_metrics["f1-score"],
                    "support": class_metrics["support"]
                }

                all_classification_reports.append(report_row)

        print(f"\n{model_name}")
        print(f"Best params: {rf_best_params[prefix] if model_name == 'Random Forest' else xgb_best_params[prefix]}")
        print(f"ROC AUC: {metrics['roc_auc']:.3f}")
        print(f"Precision: {metrics['precision']:.3f}")
        print(f"Recall: {metrics['recall']:.3f}")
        print(f"Accuracy: {metrics['accuracy']:.3f}")
        print(f"F1: {metrics['f1']:.3f}")
        print("Confusion matrix:")
        print(cm)


Final test-set validation for: mask

Random Forest
Best params: {'criterion': 'entropy', 'max_depth': 20, 'max_features': 'sqrt', 'min_samples_leaf': 1, 'min_samples_split': 2}
ROC AUC: 0.900
Precision: 0.849
Recall: 0.831
Accuracy: 0.829
F1: 0.840
Confusion matrix:
[[3056  640]
 [ 734 3598]]

XGBoost
Best params: {'colsample_bytree': 0.8, 'gamma': 0.1, 'learning_rate': 0.05, 'max_depth': 7, 'min_child_weight': 1, 'reg_lambda': 1.0, 'subsample': 0.8}
ROC AUC: 0.903
Precision: 0.851
Recall: 0.840
Accuracy: 0.834
F1: 0.845
Confusion matrix:
[[3058  638]
 [ 694 3638]]

Final test-set validation for: protective

Random Forest
Best params: {'criterion': 'entropy', 'max_depth': 20, 'max_features': 'sqrt', 'min_samples_leaf': 1, 'min_samples_split': 2}
ROC AUC: 0.833
Precision: 0.824
Recall: 0.805
Accuracy: 0.761
F1: 0.814
Confusion matrix:
[[1899  899]
 [1020 4210]]

XGBoost
Best params: {'colsample_bytree': 0.6, 'gamma': 0.1, 'learning_rate': 0.05, 'max_depth': 7, 'min_child_weight': 1, 'r

In [ ]:
# cell 6
# Save final outputs
test_results_df = pd.DataFrame(all_test_results)

test_results_df = test_results_df.sort_values(
    by=["outcome", "roc_auc", "f1"],
    ascending=[True, False, False]
).reset_index(drop=True)

confusion_matrix_df = pd.DataFrame(all_confusion_matrices)

classification_report_df = pd.DataFrame(all_classification_reports)

test_results_df.to_csv(
    f"{RESULTS_OUTPUT}\\final_test_performance.csv",
    index=False
)

confusion_matrix_df.to_csv(
    f"{RESULTS_OUTPUT}\\final_test_confusion_matrices.csv",
    index=False
)

classification_report_df.to_csv(
    f"{RESULTS_OUTPUT}\\final_test_classification_reports.csv",
    index=False
)

test_results_df

,model,outcome,upsample,precision,recall,roc_auc,accuracy,f1
0,XGBoost,mask,True,0.850795,0.839797,0.902726,0.834081,0.845260
1,Random Forest,mask,True,0.848985,0.830563,0.900394,0.828849,0.839673
2,XGBoost,protective,True,0.847367,0.760038,0.833926,0.754484,0.801331
3,Random Forest,protective,True,0.824036,0.804971,0.832914,0.760962,0.814392
4,XGBoost,social_avoidance,True,0.792851,0.729958,0.808587,0.727952,0.760105
5,Random Forest,social_avoidance,True,0.786295,0.755274,0.807518,0.734305,0.770472
